In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# Text data
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf=8")

In [7]:
document = loader.load()

In [8]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='\ufeffPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, 

In [9]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research1.pdf")

# document = pdf_loader.load()
# document

In [10]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research1.pdf")

# document = pdf_loader.load()
# document

# Ingestion Pipeline

In [11]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [12]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):

            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [13]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [14]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [15]:
# chunks
# !pip install langchain_text_splitters

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )
    
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [17]:
chunks = split_docs(all_pdf_documents)

In [18]:
len(chunks)

320

### Embedding

In [19]:
from sentence_transformers import SentenceTransformer

In [20]:
class Embeddingmanager():
    def __init__(self, model_name="all-MiniLM-L6-v2"):

        self.model_name = model_name
        print("loading model....", self.model_name)
        
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [21]:
embedding_manager = Embeddingmanager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


### Vector Store

In [22]:
import chromadb
import uuid

In [23]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
    
        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())


    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata

        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
    
        print("total documents added in vector store =", len(documents_content))
        print("docs in collection:", self.collection.count())

In [24]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [25]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding) 

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape: (320, 384)
total documents added in vector store = 320
docs in collection: 320


# Retrieval Pipeline

In [26]:
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semanntic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k,
        )

        # cosine similarity
        retrieved_docs=[]
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id" : doc_id,
                        "document" : document,
                        "metadata" : metadata,
                        "distance" : distance,
                        "similarity_score" : similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

                

In [28]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [29]:
rag_retriever.retrieve("what is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 4 documents


[{'id': 'doc_e087558f-b7a0-42e8-a857-63695be969f8',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': {'title': 'Attention is All you Need',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-to-Fr

In [30]:
rag_retriever.retrieve("what is RAG")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_1590dd9f-9bdb-4802-8322-d6835e49a0d5',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'source': 'data/pdfs\\research2.pdf',
   'creator': 'LaTeX with hyperref',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'page_label': '1',
   'keywords': '',
   'doc_index': 87,
   'page': 0,
   'author': '',
   'subject': '',
   'title': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'producer': 'pdfTeX-1.40.25',
   'trapped': '/False',
   'content_length': 288,
   'total_pages': 21},
  'distance': 0.4226756691932678,
  'similarity_score': 0.5773243308067322,
  'rank': 1},
 {'id': 'doc_5ec557ed-

# Integrate with LLMs

## Groq

In [31]:
# !pip install langchain-groq

In [32]:
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY_GROQ = os.getenv("GROQ_API_KEY")

In [33]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model_name="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [34]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relavent context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context = {context}
                Query = {query} """

    response = llm.invoke([prompt.format(context=context, query=query)])   # expecting a list as prompt
    return response.content

In [35]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [36]:
print(answer)

<think>
Okay, the user is asking "what is encoder-decoder?" and I need to use the provided context to answer. Let me start by reading through the context carefully.

The context mentions that both the encoder and decoder are stacks of N=6 identical layers. The encoder has two sub-layers in each layer: multi-head self-attention and a position-wise feed-forward network. The decoder has three sub-layers, adding a third one that does multi-head attention over the encoder's output. Both use residual connections and layer normalization. The base model uses a dropout rate of 0.1. The encoder and decoder stacks also have positional encodings.

So, the encoder-decoder architecture here is likely referring to the Transformer model. The encoder processes input data through multiple layers with self-attention, and the decoder generates output, using the encoder's output for attention. The key points to include are the number of layers, the sub-layers in each, the attention mechanisms, residual con

In [38]:
## OpenAI : 
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini", api_key="...")

In [39]:
## Anthropic : 
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-3-haiku-20240307", api_key="...")